In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
data["face"].shape

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
len(data_classified[1])

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


# PCA_funciton

In [ ]:
import numpy as np

def compute_pca(data_matrix, num_components=None, variance_threshold=None):
    """
    Perform PCA on a data matrix (samples × features).
    
    Parameters:
        data_matrix: np.ndarray, shape (n_samples, n_features)
        num_components: int or None — number of components to keep
        variance_threshold: float or None — keep components that explain up to this cumulative variance (e.g., 0.95)
    
    Returns:
        pca_result: projected data, shape (n_samples, num_components)
        components: principal components (eigenvectors)
        explained_variance_ratio: array of variance explained by each component
        mean: mean of original data (for inverse transform if needed)
    """
    # Step 1: Center the data
    mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - mean

    # Step 2: Covariance matrix
    cov_matrix = np.cov(centered_data, rowvar=False)

    # Step 3: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    # Step 4: Sort in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 5: Compute explained variance
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Step 6: Determine number of components
    if variance_threshold is not None:
        num_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Using {num_components} components to explain {variance_threshold*100:.1f}% variance.")
    elif num_components is None:
        num_components = data_matrix.shape[1]  # Keep all

    # Step 7: Select components and project
    selected_components = eigenvectors[:, :num_components]
    pca_result = np.dot(centered_data, selected_components)

    return pca_result, selected_components, explained_variance_ratio[:num_components], mean


# MDA Function

In [ ]:
import numpy as np

def compute_mda(data_matrix, labels, num_components=None):
    """
    Perform MDA (also known as LDA) on the given data.

    Parameters:
        data_matrix: np.ndarray of shape (n_samples, n_features)
        labels: array-like of shape (n_samples,)
        num_components: int or None — number of components to retain (must be ≤ n_classes - 1)

    Returns:
        mda_result: Projected data of shape (n_samples, num_components)
        components: Eigenvectors used for projection (n_features, num_components)
        eigenvalues: Corresponding eigenvalues
        overall_mean: Mean of the original data
    """
    n_samples, n_features = data_matrix.shape
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)

    # Step 1: Center the data
    overall_mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - overall_mean

    # Step 2: Compute class means
    class_means = []
    for c in unique_classes:
        class_data = data_matrix[labels == c]
        class_mean = np.mean(class_data, axis=0)
        class_means.append(class_mean)

    # Step 3: Compute between-class scatter matrix (S_B)
    S_B = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        n_i = np.sum(labels == c)
        mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
        S_B += (n_i / n_samples) * (mean_diff @ mean_diff.T)

    # Step 4: Compute within-class scatter matrix (S_W)
    S_W = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        class_data = data_matrix[labels == c]
        n_i = class_data.shape[0]
        class_centered = class_data - class_means[i]
        S_W += (n_i / n_samples) * (class_centered.T @ class_centered) / n_i

    # Step 5: Solve generalized eigenvalue problem
    S_W_inv = np.linalg.pinv(S_W)
    eig_matrix = S_W_inv @ S_B
    eigenvalues, eigenvectors = np.linalg.eigh(eig_matrix)

    # Step 6: Sort eigenvectors by eigenvalue magnitude (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 7: Limit number of components
    max_components = n_classes - 1
    if num_components is None or num_components > max_components:
        num_components = max_components
        print(f"Using max possible components for MDA: {num_components}")

    selected_components = eigenvectors[:, :num_components]
    mda_result = centered_data @ selected_components

    return mda_result, selected_components, eigenvalues[:num_components], overall_mean


# data seperation

In [ ]:
def separate_train_test_manual(data, labels, sub_num_total, train_num, task_num, random_state=42):
    np.random.seed(random_state)
    
    if task_num == 1:  # Person identification with all 3 images
        selected_subjects = np.random.choice(sub_num_total, train_num, replace=False)
        train_indices = []
        test_indices = []

        # For each subject, put 2 images in training and 1 in testing
        for s in selected_subjects:
            base = 3 * s
            train_indices.extend([base, base + 1])  # First 2 images to training
            test_indices.append(base + 2)           # Last image to testing

        train_set = data[train_indices]
        train_labels = labels[train_indices]
        test_set = data[test_indices]
        test_labels = labels[test_indices]

    elif task_num == 2:
        # Get indices for neutral and expression images
        neutral_indices = list(range(0, 3 * sub_num_total, 3))     # Images at positions 0, 3, 6, ...
        expression_indices = list(range(1, 3 * sub_num_total, 3))  # Images at positions 1, 4, 7, ...
        
        # Create binary labels (0 for neutral, 1 for expression)
        neutral_labels = np.zeros(len(neutral_indices), dtype=int)
        expression_labels = np.ones(len(expression_indices), dtype=int)
        
        # Shuffle each set of indices separately
        np.random.shuffle(neutral_indices)
        np.random.shuffle(expression_indices)
        
        # Split each class into training (80%) and testing (20%)
        neutral_split = int(len(neutral_indices) * 0.8)
        expression_split = int(len(expression_indices) * 0.8)
        
        # Create training and testing sets for each class
        neutral_train = neutral_indices[:neutral_split]
        neutral_test = neutral_indices[neutral_split:]
        expression_train = expression_indices[:expression_split]
        expression_test = expression_indices[expression_split:]
        
        # Combine indices and labels
        train_indices = np.concatenate([neutral_train, expression_train])
        test_indices = np.concatenate([neutral_test, expression_test])
        
        # Create labels matching the indices
        train_labels = np.concatenate([np.zeros(len(neutral_train)), np.ones(len(expression_train))])
        test_labels = np.concatenate([np.zeros(len(neutral_test)), np.ones(len(expression_test))])
        
        # Extract the data
        train_set = data[train_indices]
        test_set = data[test_indices]
    
    else:
        raise ValueError("❌ Invalid task_num. Use 1 for person ID or 2 for expression classification.")
    return train_set, train_labels, test_set, test_labels


# Kernel SVM

In [ ]:
import numpy as np
from cvxopt import matrix, solvers

# -----------------------------
# Kernels
# -----------------------------

def rbf_kernel(x, y, sigma=1.0):
    return np.exp(-np.linalg.norm(x - y) ** 2 / (2 * sigma ** 2))

def poly_kernel(x, y, degree=3):
    return (np.dot(x, y) + 1) ** degree

def poly_kernel_reverse(x, y, degree=3):
    return (np.dot(x, y) + 1) ** (1/degree)

def linear_kernel (x,y):
    return np.dot (x,y)

# -----------------------------
# Kernel SVM Classifier (with optional C)
# -----------------------------

class KernelSVM:
    def __init__(self, C=None, kernel='rbf', sigma=1.0, degree=3):
        self.C = C  # None means hard-margin
        self.kernel_type = kernel
        self.sigma = sigma
        self.degree = degree
        self.kernel = self._get_kernel(kernel)

    def _get_kernel(self, kernel_type):
        if kernel_type == 'rbf':
            return lambda x, y: rbf_kernel(x, y, self.sigma)
        elif kernel_type == 'poly':
            return lambda x, y: poly_kernel(x, y, self.degree)
        elif kernel_type == 'poly_rev':
            return lambda x, y: poly_kernel_reverse(x, y, self.degree)
        elif kernel_type == 'linear':
            return lambda x, y: linear_kernel(x, y)
        else:
            raise ValueError("Unsupported kernel.")

    def fit(self, X, y):
        n_samples, _ = X.shape
        K = np.zeros((n_samples, n_samples))

        # Compute kernel matrix
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.kernel(X[i], X[j])

        # QP problem matrices
        P = matrix(np.outer(y, y) * K)
        q = matrix(-np.ones(n_samples))
        A = matrix(y.reshape(1, -1).astype('double'))
        b = matrix(np.zeros(1))

        # Choose G and h based on whether C is None
        if self.C is None:
            # Hard margin: alpha_i ≥ 0
            G = matrix(-np.eye(n_samples))
            h = matrix(np.zeros(n_samples))
        else:
            # Soft margin: 0 ≤ alpha_i ≤ C
            G = matrix(np.vstack((
                -np.eye(n_samples),           # -alpha_i ≤ 0 → alpha_i ≥ 0
                 np.eye(n_samples)            # alpha_i ≤ C
            )))
            h = matrix(np.hstack((
                np.zeros(n_samples),
                np.ones(n_samples) * self.C
            )))

        # Solve QP problem
        solvers.options['show_progress'] = False
        solution = solvers.qp(P, q, G, h, A, b)
        alphas = np.ravel(solution['x'])

        # Support vectors have non-zero alphas
        sv = alphas > 1e-5
        self.alphas = alphas[sv]
        self.support_vectors = X[sv]
        self.support_vector_labels = y[sv]

        # Bias term: averaged over support vectors
        self.b = np.mean([
            y_k - np.sum(self.alphas * self.support_vector_labels *
                         np.array([self.kernel(x_k, x_j) for x_j in self.support_vectors]))
            for (x_k, y_k) in zip(self.support_vectors, self.support_vector_labels)
        ])

    def project(self, X):
        y_pred = []
        for x in X:
            result = np.sum([
                a * y_sv * self.kernel(x, x_sv)
                for a, y_sv, x_sv in zip(self.alphas, self.support_vector_labels, self.support_vectors)
            ])
            y_pred.append(result + self.b)
        return np.array(y_pred)

    def predict(self, X):
        return np.sign(self.project(X))


In [ ]:


train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


# Convert labels from [0, 1] to [-1, 1]
train_labels = 2 * train_labels - 1
test_labels = 2 * test_labels - 1

In [ ]:
## svm _qopt
import numpy as np
import matplotlib.pyplot as plt

sigma_values = np.arange(0.1, 20, 0.5)
train_accuracies = []
test_accuracies = []

for sigma in sigma_values:
    svm_rbf = KernelSVM(C=1.0, kernel='rbf', sigma=sigma)
    svm_rbf.fit(train_set, train_labels)

    # Training accuracy
    train_pred = svm_rbf.predict(train_set)
    train_acc = np.mean(train_pred == train_labels) * 100
    train_accuracies.append(train_acc)

    # Test accuracy
    pred_rbf = svm_rbf.predict(test_set)
    acc_rbf = np.mean(pred_rbf == test_labels) * 100
    test_accuracies.append(acc_rbf)

    print(f"Sigma: {sigma:.2f} | Train Acc: {train_acc:.2f}% | Test Acc: {acc_rbf:.2f}%")

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(sigma_values, train_accuracies, label='Train Accuracy', marker='o')
plt.plot(sigma_values, test_accuracies, label='Test Accuracy', marker='s')
plt.axvline(sigma_values[np.argmax(test_accuracies)], color='gray', linestyle='--', label='Best σ')
plt.xlabel("Sigma (RBF width)")
plt.ylabel("Accuracy (%)")
plt.title("Train/Test Accuracy vs. RBF Kernel Sigma")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## svm -qpt polynomial
import numpy as np
import matplotlib.pyplot as plt

# Define degree range for the polynomial kernel
degrees = np.arange(1, 10)
train_accuracies_poly = []
test_accuracies_poly = []

# Loop over degrees and train/predict
for d in degrees:
    svm_poly = KernelSVM(C=1.0, kernel='poly', degree=d)
    svm_poly.fit(train_set, train_labels)

    # Training accuracy
    train_pred = svm_poly.predict(train_set)
    train_acc = np.mean(train_pred == train_labels) * 100
    train_accuracies_poly.append(train_acc)

    # Test accuracy
    pred_poly = svm_poly.predict(test_set)
    acc_poly = np.mean(pred_poly == test_labels) * 100
    test_accuracies_poly.append(acc_poly)

    print(f"Degree: {d} | Train Acc: {train_acc:.2f}% | Test Acc: {acc_poly:.2f}%")

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(degrees, train_accuracies_poly, label='Train Accuracy', marker='o')
plt.plot(degrees, test_accuracies_poly, label='Test Accuracy', marker='s')
plt.axvline(degrees[np.argmax(test_accuracies_poly)], color='gray', linestyle='--', label='Best degree')
plt.xlabel("Polynomial Degree")
plt.ylabel("Accuracy (%)")
plt.title("Train/Test Accuracy vs. Polynomial Kernel Degree")
plt.legend()
plt.grid(True)
plt.show()


# kernel svm gradient descent

In [ ]:
class KernelSVM_GD:
    def __init__(self, C=1.0, kernel='rbf', sigma=1.0, degree=3, lr=0.001, max_iter=1000):
        self.C = C
        self.sigma = sigma
        self.degree = degree
        self.kernel_type = kernel
        self.lr = lr
        self.max_iter = max_iter
        self.kernel = self._get_kernel(kernel)

    def _get_kernel(self, kernel_type):
        if kernel_type == 'rbf':
            return lambda x, y: rbf_kernel(x, y, self.sigma)
        elif kernel_type == 'poly':
            return lambda x, y: poly_kernel(x, y, self.degree)
        elif kernel_type == 'poly_rev':
            return lambda x, y: poly_kernel_reverse(x, y, self.degree)
        elif kernel_type == 'linear':
            return lambda x, y: linear_kernel(x, y)
        else:
            raise ValueError("Unsupported kernel.")

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.X = X
        self.y = y
        self.alphas = np.zeros(n_samples)

        # Precompute full kernel matrix
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i, j] = self.kernel(X[i], X[j])

        # Dual objective: maximize L = sum αi - 1/2 sum_i,j αi αj yi yj K(xi, xj)
        for it in range(self.max_iter):
            for i in range(n_samples):
                # Gradient of dual L wrt α_i
                gradient = 1 - np.sum(
                    self.alphas * y * y[i] * K[:, i]
                )
                self.alphas[i] += self.lr * gradient

                # Project α back into bounds [0, C]
                self.alphas[i] = np.clip(self.alphas[i], 0, self.C)

        # Support vectors
        sv = self.alphas > 1e-5
        self.support_vectors = X[sv]
        self.support_vector_labels = y[sv]
        self.alphas = self.alphas[sv]

        # Bias: average over support vectors
        self.b = np.mean([
            y_i - np.sum(self.alphas * self.support_vector_labels *
                         np.array([self.kernel(x_i, x_j) for x_j in self.support_vectors]))
            for x_i, y_i in zip(self.support_vectors, self.support_vector_labels)
        ])

    def project(self, X):
        y_pred = []
        for x in X:
            result = np.sum([
                a * y_sv * self.kernel(x, x_sv)
                for a, y_sv, x_sv in zip(self.alphas, self.support_vector_labels, self.support_vectors)
            ])
            y_pred.append(result + self.b)
        return np.array(y_pred)

    def predict(self, X):
        return np.sign(self.project(X))


In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


# Convert labels from [0, 1] to [-1, 1]
train_labels = 2 * train_labels - 1
test_labels = 2 * test_labels - 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Store accuracies
sigma_values = np.arange(1, 10)
poly_degrees = np.arange(1, 10)

rbf_train_accuracies = []
rbf_test_accuracies = []

poly_train_accuracies = []
poly_test_accuracies = []

# RBF Kernel
print("📡 Testing RBF Kernel with Gradient Descent")
for sigma in sigma_values:
    svm_rbf = KernelSVM_GD(kernel='rbf', sigma=sigma, C=1.0, lr=0.001, max_iter=500)
    svm_rbf.fit(train_set, train_labels)

    train_pred = svm_rbf.predict(train_set)
    train_acc = np.mean(train_pred == train_labels) * 100
    rbf_train_accuracies.append(train_acc)

    test_pred = svm_rbf.predict(test_set)
    test_acc = np.mean(test_pred == test_labels) * 100
    rbf_test_accuracies.append(test_acc)

    print(f"σ={sigma} → Train: {train_acc:.2f}%, Test: {test_acc:.2f}%")

# Polynomial Kernel
print("\n🔢 Testing Polynomial Kernel with Gradient Descent")
for degree in poly_degrees:
    svm_poly = KernelSVM_GD(kernel='poly', degree=degree, C=1.0, lr=0.001, max_iter=500)
    svm_poly.fit(train_set, train_labels)

    train_pred = svm_poly.predict(train_set)
    train_acc = np.mean(train_pred == train_labels) * 100
    poly_train_accuracies.append(train_acc)

    test_pred = svm_poly.predict(test_set)
    test_acc = np.mean(test_pred == test_labels) * 100
    poly_test_accuracies.append(test_acc)

    print(f"degree={degree} → Train: {train_acc:.2f}%, Test: {test_acc:.2f}%")

# Plot RBF Kernel results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(sigma_values, rbf_train_accuracies, marker='o', label='Train Accuracy')
plt.plot(sigma_values, rbf_test_accuracies, marker='s', label='Test Accuracy')
plt.title('RBF Kernel: Accuracy vs Sigma')
plt.xlabel('Sigma (σ)')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

# Plot Polynomial Kernel results
plt.subplot(1, 2, 2)
plt.plot(poly_degrees, poly_train_accuracies, marker='o', label='Train Accuracy')
plt.plot(poly_degrees, poly_test_accuracies, marker='s', label='Test Accuracy')
plt.title('Polynomial Kernel: Accuracy vs Degree')
plt.xlabel('Degree')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


# cross validation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to perform K-fold split manually
def k_fold_split(X, y, k=5, seed=42):
    np.random.seed(seed)
    n_samples = len(X)
    indices = np.random.permutation(n_samples)
    fold_sizes = np.full(k, n_samples // k)
    fold_sizes[:n_samples % k] += 1
    current = 0
    splits = []
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        val_idx = indices[start:stop]
        train_idx = np.concatenate((indices[:start], indices[stop:]))
        splits.append((train_idx, val_idx))
        current = stop
    return splits

# --- Setup ---
k = 5  # number of folds
sigma_values = np.arange(1, 10)
poly_degrees = np.arange(1, 10)

rbf_cv_accuracies = []
poly_cv_accuracies = []

splits = k_fold_split(train_set, train_labels, k=k)

# --- RBF Kernel ---
print("📡 RBF Kernel Cross-Validation")
for sigma in sigma_values:
    fold_accuracies = []
    for train_idx, val_idx in splits:
        X_train, X_val = train_set[train_idx], train_set[val_idx]
        y_train, y_val = train_labels[train_idx], train_labels[val_idx]

        svm = KernelSVM_GD(kernel='rbf', sigma=sigma, C=1.0, lr=0.001, max_iter=500)
        svm.fit(X_train, y_train)
        val_pred = svm.predict(X_val)
        acc = np.mean(val_pred == y_val) * 100
        fold_accuracies.append(acc)

    avg_acc = np.mean(fold_accuracies)
    rbf_cv_accuracies.append(avg_acc)
    print(f"σ={sigma} → CV Accuracy: {avg_acc:.2f}%")

# --- Polynomial Kernel ---
print("\n🔢 Polynomial Kernel Cross-Validation")
for degree in poly_degrees:
    fold_accuracies = []
    for train_idx, val_idx in splits:
        X_train, X_val = train_set[train_idx], train_set[val_idx]
        y_train, y_val = train_labels[train_idx], train_labels[val_idx]

        svm = KernelSVM_GD(kernel='poly', degree=degree, C=1.0, lr=0.001, max_iter=500)
        svm.fit(X_train, y_train)
        val_pred = svm.predict(X_val)
        acc = np.mean(val_pred == y_val) * 100
        fold_accuracies.append(acc)

    avg_acc = np.mean(fold_accuracies)
    poly_cv_accuracies.append(avg_acc)
    print(f"degree={degree} → CV Accuracy: {avg_acc:.2f}%")

# --- Plotting ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(sigma_values, rbf_cv_accuracies, marker='o')
plt.title('RBF Kernel CV Accuracy vs Sigma')
plt.xlabel('Sigma (σ)')
plt.ylabel('CV Accuracy (%)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(poly_degrees, poly_cv_accuracies, marker='o')
plt.title('Polynomial Kernel CV Accuracy vs Degree')
plt.xlabel('Degree')
plt.ylabel('CV Accuracy (%)')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


# Convert labels from [0, 1] to [-1, 1]
train_labels = 2 * train_labels - 1
test_labels = 2 * test_labels - 1

In [ ]:
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [2]

# Store accuracies separately
rbf_pca_acc, poly_pca_acc = [], []
rbf_mda_acc, poly_mda_acc = [], []

# Loop over PCA components
for n in pca_components_list:
    X_train_pca, pca_comp, _, mean_vec = compute_pca(train_set, num_components=n)
    X_test_pca = np.dot(test_set - mean_vec, pca_comp)

    # RBF kernel (sigma=9)
    svm_rbf = KernelSVM(C=1.0, kernel='rbf', sigma=9)
    svm_rbf.fit(X_train_pca, train_labels)
    acc_rbf = np.mean(svm_rbf.predict(X_test_pca) == test_labels) * 100
    rbf_pca_acc.append(acc_rbf)

    # Polynomial kernel (degree=3)
    svm_poly = KernelSVM(C=1.0, kernel='poly', degree=3)
    svm_poly.fit(X_train_pca, train_labels)
    acc_poly = np.mean(svm_poly.predict(X_test_pca) == test_labels) * 100
    poly_pca_acc.append(acc_poly)

# Loop over MDA components
for n in mda_components_list:
    X_train_mda, mda_comp, _, mean_vec = compute_mda(train_set, train_labels, num_components=n)
    X_test_mda = np.dot(test_set - mean_vec, mda_comp)

    # RBF kernel (sigma=9)
    svm_rbf = KernelSVM(C=1.0, kernel='rbf', sigma=9)
    svm_rbf.fit(X_train_mda, train_labels)
    acc_rbf = np.mean(svm_rbf.predict(X_test_mda) == test_labels) * 100
    rbf_mda_acc.append(acc_rbf)

    # Polynomial kernel (degree=3)
    svm_poly = KernelSVM(C=1.0, kernel='poly', degree=3)
    svm_poly.fit(X_train_mda, train_labels)
    acc_poly = np.mean(svm_poly.predict(X_test_mda) == test_labels) * 100
    poly_mda_acc.append(acc_poly)


# MDA + PCA

In [ ]:


train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


# Convert labels from [0, 1] to [-1, 1]
train_labels = 2 * train_labels - 1
test_labels = 2 * test_labels - 1

In [ ]:
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,          # <- Or mda_result or flattened_data
    labels=person_labels,     # <- Label array (1D)
    sub_num_total=200,        # <- Total number of subjects
    train_num=200,            # <- Total samples to use for training (adjust if needed)
    task_num=2                # <- Use 1 for identity, 2 for expression classification
)


# Convert labels from [0, 1] to [-1, 1]
train_labels = 2 * train_labels - 1
test_labels = 2 * test_labels - 1

In [ ]:
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [2]

poly_pca_acc, poly_mda_acc = [], []
rbf_pca_acc, rbf_mda_acc = [], []

# Loop over PCA components
for n in pca_components_list:
    X_train_pca, pca_comp, _, mean_vec = compute_pca(train_set, num_components=n)
    X_test_pca = np.dot(test_set - mean_vec, pca_comp)

    svm_poly = KernelSVM_GD(kernel='poly', degree=3, C=1.0, lr=0.001, max_iter=500)
    svm_poly.fit(X_train_pca, train_labels)
    acc_poly = np.mean(svm_poly.predict(X_test_pca) == test_labels) * 100
    poly_pca_acc.append(acc_poly)

    svm_rbf = KernelSVM_GD(kernel='rbf', sigma=9, C=1.0, lr=0.001, max_iter=500)
    svm_rbf.fit(X_train_pca, train_labels)
    acc_rbf = np.mean(svm_rbf.predict(X_test_pca) == test_labels) * 100
    rbf_pca_acc.append(acc_rbf)

# Loop over MDA components
for n in mda_components_list:
    X_train_mda, mda_comp, _, mean_vec = compute_mda(train_set, train_labels, num_components=n)
    X_test_mda = np.dot(test_set - mean_vec, mda_comp)

    svm_poly = KernelSVM_GD(kernel='poly', degree=3, C=1.0, lr=0.001, max_iter=500)
    svm_poly.fit(X_train_mda, train_labels)
    acc_poly = np.mean(svm_poly.predict(X_test_mda) == test_labels) * 100
    poly_mda_acc.append(acc_poly)

    svm_rbf = KernelSVM_GD(kernel='rbf', sigma=9, C=1.0, lr=0.001, max_iter=500)
    svm_rbf.fit(X_train_mda, train_labels)
    acc_rbf = np.mean(svm_rbf.predict(X_test_mda) == test_labels) * 100
    rbf_mda_acc.append(acc_rbf)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(pca_components_list, rbf_pca_acc, marker='o', label='RBF + PCA')
plt.axhline( rbf_mda_acc[0], linestyle='--', color='orange', label=f'RBF + MDA ({rbf_mda_acc[0]})')
plt.axhline(91.25 ,linestyle='-.', color='black', label=f'RBF for original data ')
plt.title("RBF Kernel")
plt.xlabel("Components")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(pca_components_list, poly_pca_acc, marker='o', label='Poly + PCA')
plt.axhline( poly_mda_acc[0], linestyle='--', color='orange', label=f'Poly + MDA ({poly_mda_acc[0]})')
plt.axhline(91.25 ,linestyle='-.', color='black', label=f'polynomial for original data ')
plt.title("Polynomial Kernel")
plt.xlabel("Components")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()
